# Second-Corpus Generality Check (Phase 6)

Does the QASPER negative result — *cheap document features don't predict per-document optimal leaf size* — **generalize to another corpus**? QASPER's gold-evidence proxy isn't available on QuALITY/NarrativeQA, so we use an **answer-recall** proxy: for each question, the best token-recall of any gold answer string inside the retrieved context (`granularity_sweep.answer_coverage`). The per-doc optimal size is the size maximizing mean answer-recall.

**Default corpus = QuALITY** (passages are short ⇒ free-tier safe). Switch `CORPUS='narrativeqa'` ONLY if you have more compute (full stories are large to download + embed). **No LLM anywhere.**

**Caveat (state in the paper):** answer-recall is a weaker proxy than QASPER's gold-evidence coverage (gold answers are often paraphrases). We therefore report proxy-sanity diagnostics (does H1 headroom even exist?) before reading the feature gate. This is a *directional* generality check.


In [ ]:
# 1) Code.
!git clone -b ckraptor https://github.com/MissLostCodes/raptor.git
%cd raptor
!pip install -q -r requirements-colab.txt

In [ ]:
# 2) Config + Drive cache. Conservative caps keep this inside the free tier.
CORPUS = 'quality'        # 'quality' (free-tier safe) | 'narrativeqa' (needs more compute)
SPLIT = 'validation'      # a split WITH gold labels (QuALITY test withholds them)
N_DOCS = 30               # cap docs
MAX_Q_PER_DOC = 15        # cap questions/doc (keeps retrieval count bounded)
SIZES = [50, 100, 150, 200, 300, 400]

try:
    from google.colab import drive
    drive.mount('/content/drive')
    RUN_DIR = '/content/drive/MyDrive/raptor_runs'
except Exception as e:
    print('Not on Colab / Drive unavailable -> ./raptor_runs', e)
    RUN_DIR = 'raptor_runs'
import os
os.makedirs(RUN_DIR, exist_ok=True)
OUT = os.path.join(RUN_DIR, f'{CORPUS}_answercov_sweep.json')   # resumable cache
SUMMARY = os.path.join(RUN_DIR, f'{CORPUS}_second_corpus_summary.json')
print('CORPUS=', CORPUS, '| OUT=', OUT, '(exists:', os.path.exists(OUT), ')')
try:
    import torch
    print('CUDA:', torch.cuda.is_available(),
          '|', (torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'))
except Exception as e:
    print('torch not ready:', e)

In [ ]:
# 3) Load corpus and build a uniform gold-answer target.
#    QuALITY: gold answer = the correct option's text (options[gold_index]).
#    NarrativeQA: gold answers = the reference answers (already populated).
from experiments.datasets import get_loader

if CORPUS == 'quality':
    docs = get_loader('quality', split=SPLIT).load(limit=None)
    for d in docs:
        for q in d.questions:
            if q.gold_index is not None and q.options and 0 <= q.gold_index < len(q.options):
                q.gold_answers = [q.options[q.gold_index]]
            else:
                q.gold_answers = []
elif CORPUS == 'narrativeqa':
    docs = get_loader('narrativeqa').load(limit=None)
else:
    raise ValueError(CORPUS)

# Keep only docs with scored questions; apply caps.
clean = []
for d in docs:
    d.questions = [q for q in d.questions if [a for a in (q.gold_answers or []) if a and a.strip()]]
    if d.questions:
        d.questions = d.questions[:MAX_Q_PER_DOC]
        clean.append(d)
docs = clean[:N_DOCS]

import statistics
doclens = [len(d.text.split()) for d in docs]
print(f'{len(docs)} {CORPUS} docs after caps; '
      f'{sum(len(d.questions) for d in docs)} scored questions')
print(f'doc length (tokens): median={int(statistics.median(doclens))} '
      f'min={min(doclens)} max={max(doclens)}')

In [ ]:
# 4) Run the sweep with the ANSWER-RECALL proxy (resumable; no LLM).
from experiments import granularity_sweep as gs, granularity_calibration as gc
from collections import Counter
from tqdm.auto import tqdm

records = gs.run_sweep(
    docs, SIZES, budget=2000, out_path=OUT,
    coverage_fn=gs._answer_coverage_q,                 # <-- the generalization
    progress=lambda m: tqdm.write(m),
)
print(f'{len(records)} (doc, size) records')

# --- proxy sanity + H1 headroom: does adaptive headroom even EXIST here? ---
h = gc.headroom(records)
print(f"\nH1 headroom: best fixed = {h['best_fixed_size']} tok @ {h['best_fixed_cov']:.4f} | "
      f"oracle {h['oracle_cov']:.4f} | +{h['rel_gain_pct']:.1f}% rel.")
opt = gs.best_size_per_doc(records)
print(f'{len(opt)} docs with a defined optimal size')
print('optimal-size distribution:', dict(sorted(Counter(opt.values()).items())))
mean_cov = statistics.mean([r['mean_evidence_coverage'] for r in records
                            if r['mean_evidence_coverage'] is not None])
print(f'mean answer-recall across (doc,size): {mean_cov:.3f}  '
      f'(near 0 or near 1 => proxy is degenerate; interpret the gate with care)')

In [ ]:
# 5) Features: surface (the H2 six) + locality (A1-A5,B1,C4). One batched SBERT pass/doc.
import time
from raptor.EmbeddingModels import SBertEmbeddingModel
from raptor.chunking.density_score import density_score
from raptor.chunking.segment_features import segment_features, split_sentences

_sbert = SBertEmbeddingModel()._ensure_model()
def batched_embed(sents):
    return _sbert.encode(sents, batch_size=64, convert_to_numpy=True,
                         normalize_embeddings=True, show_progress_bar=False)

SURFACE = ['mean_sentence_len_tokens_norm','type_token_ratio','numeral_symbol_density',
           'mean_word_len_chars_norm','list_marker_density','nonstopword_ratio']
LOCALITY = ['mean_segment_len_tokens','paragraph_len_mean','paragraph_len_cv',
            'var_adjacent_cosine','topic_shift_rate','mean_adjacent_cosine',
            'segment_len_cv','gzip_ratio']

feats = {}
t0 = time.time()
for d in tqdm(docs, desc='features'):
    f = dict(density_score(d.text)[1])
    f.update(segment_features(d.text, embed_fn=batched_embed))
    feats[d.doc_id] = f
print(f'features for {len(feats)} docs in {time.time()-t0:.1f}s')

In [ ]:
# 6) Feature gate — does the NEGATIVE replicate on this corpus?
ids = [k for k in opt if k in feats]
opt_vec = [opt[k] for k in ids]
ALL_FEATURES = SURFACE + LOCALITY
print(f"{'feature':<30}{'rho vs OPT size':>16}")
print('-' * 46)
rows = [(f, gc.rank_corr([feats[k][f] for k in ids], opt_vec)) for f in ALL_FEATURES]
for f, r in sorted(rows, key=lambda t: abs(t[1]), reverse=True):
    flag = '  <==|rho|>=0.5' if abs(r) >= 0.5 else ('  ~marg' if abs(r) >= 0.4 else '')
    print(f'{f:<30}{r:>16.3f}{flag}')

# Held-out 50-split gate (combined surface+locality pool), reusing the H2 protocol.
import statistics
bf_size, _ = gc.best_global_fixed(records)
_, oracle_sizes_all = gc.oracle_per_doc(records)
out = {'fixed': [], 'selected feature': [], 'oracle': []}
wins = 0
for seed in tqdm(range(50), desc='held-out splits'):
    tr, te = gc.train_test_split_docs(list(opt), seed=seed)
    if not tr or not te:
        continue
    cov_fixed = gc.coverage_under_sizes(records, {k: bf_size for k in te})
    sel = gc.select_and_fit(feats, opt, tr, feature_names=ALL_FEATURES)
    pred = ({k: gc.predict_size(feats[k][sel['feature']], sel['fit'], 50, 400)
             for k in te} if sel['feature'] else {})
    cov_sel = gc.coverage_under_sizes(records, pred)
    cov_or = gc.coverage_under_sizes(records, {k: oracle_sizes_all[k] for k in te
                                               if k in oracle_sizes_all})
    out['fixed'].append(cov_fixed); out['selected feature'].append(cov_sel); out['oracle'].append(cov_or)
    if cov_sel > cov_fixed:
        wins += 1
n = len(out['fixed'])
print(f"\n{'policy (TEST coverage, %d splits)'%n:<26}{'mean':>9}{'std':>9}")
print('-' * 44)
for k, v in out.items():
    print(f'{k:<26}{statistics.mean(v):>9.4f}{statistics.pstdev(v):>9.4f}')
best_feat, best_rho = max(rows, key=lambda t: abs(t[1]))
winrate = wins / max(1, n)
replicate = (abs(best_rho) < 0.5 and wins < 30)
print(f"\nselected feature beats fixed in {wins}/{n} ({winrate:.0%}); "
      f"best |rho|={abs(best_rho):.3f} ({best_feat})")
print('NEGATIVE RESULT REPLICATES on', CORPUS, ':', replicate,
      '\n(replicates = no feature reaches |rho|>=0.5 AND selected map loses to fixed)')

In [ ]:
# 7) Persist a summary for the writeup.
import json
summary = {
    'corpus': CORPUS, 'split': SPLIT, 'n_docs': len(opt),
    'n_questions': sum(len(d.questions) for d in docs),
    'doc_len_median_tokens': int(statistics.median(doclens)),
    'mean_answer_recall': mean_cov,
    'headroom': h,
    'optimal_size_distribution': dict(sorted(Counter(opt.values()).items())),
    'feature_rho_vs_opt': {f: gc.rank_corr([feats[k][f] for k in ids], opt_vec)
                           for f in ALL_FEATURES},
    'held_out': {k: {'mean': statistics.mean(v), 'std': statistics.pstdev(v)}
                 for k, v in out.items()},
    'selected_feature_winrate': winrate,
    'best_feature': best_feat, 'best_abs_rho': abs(best_rho),
    'negative_result_replicates': bool(replicate),
}
with open(SUMMARY, 'w') as fh:
    json.dump(summary, fh, indent=2, default=float)
print('wrote', SUMMARY)
print(json.dumps(summary, indent=2, default=float))

## How to read this

1. **Proxy sanity first.** If `mean_answer_recall` is ~0 or ~1, or there is no H1 headroom, the proxy is degenerate on this corpus — say so and don't over-claim from the gate.
2. **H1 (headroom).** If a per-doc oracle still beats the best fixed size, the adaptive opportunity exists here too.
3. **The gate.** If no feature reaches `|ρ|≥0.5` and the selected-feature map still loses to a tuned fixed size, the **negative result replicates** — the refutation is not QASPER-specific.
4. Paste the printed summary (and `*_second_corpus_summary.json`) back to fold into `paper/main.tex` and a `docs/results/` writeup.
